In [1]:
import pickle
import pandas as pd
from sklearn.metrics import classification_report, roc_auc_score

# Load test data
X_train, X_test, y_train, y_test = pickle.load(open('../artifacts/preprocessed_data.pkl', 'rb'))

# Load all models and thresholds
xgb = pickle.load(open('../artifacts/xgboost_model.pkl', 'rb'))
xgb_threshold = pickle.load(open('../artifacts/threshold.pkl', 'rb'))

rf = pickle.load(open('../artifacts/rf_model.pkl', 'rb'))
rf_threshold = pickle.load(open('../artifacts/rf_threshold.pkl', 'rb'))

lr = pickle.load(open('../artifacts/logistic_model.pkl', 'rb'))
lr_threshold = pickle.load(open('../artifacts/logistic_threshold.pkl', 'rb'))

# Comparison function
def evaluate_model(name, model, threshold, X_test, y_test):
    y_proba = model.predict_proba(X_test)[:, 1]
    y_pred = (y_proba >= threshold).astype(int)
    report = classification_report(y_test, y_pred, output_dict=True)
    return {
        'Model': name,
        'Accuracy': round(report['accuracy'], 4),
        'Precision': round(report['1']['precision'], 4),
        'Recall': round(report['1']['recall'], 4),
        'F1': round(report['1']['f1-score'], 4),
        'ROC-AUC': round(roc_auc_score(y_test, y_proba), 4)
    }

results = []
results.append(evaluate_model('XGBoost', xgb, xgb_threshold, X_test, y_test))
results.append(evaluate_model('Random Forest', rf, rf_threshold, X_test, y_test))
results.append(evaluate_model('Logistic Regression', lr, lr_threshold, X_test, y_test))

df_results = pd.DataFrame(results)
print(df_results.to_string(index=False))

              Model  Accuracy  Precision  Recall     F1  ROC-AUC
            XGBoost    0.8299     0.4754  0.6170 0.5370   0.8064
      Random Forest    0.7891     0.4138  0.7660 0.5373   0.7992
Logistic Regression    0.8673     0.6250  0.4255 0.5063   0.7954
